# Step 3 — SHAP Explainability & Fairness Audit

**Inputs (produced by Steps 1 and 2):**
- The MLflow run id of the trained XGBoost model
- `data/processed/application_train_engineered.parquet` (for protected attributes used only in the audit, not as model inputs)
- `artifacts/preprocessor.joblib`
- `artifacts/feature_names.json`

**Goal:** Compute per-decision reason codes, global feature importance, and a fairness audit by protected segment (gender, family status, income decile). Log every audit artifact alongside the model so a compliance reviewer can trace any loan decision back to its model version and its explanation.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
import mlflow
import shap

from sklearn.model_selection import train_test_split

from src.utils.config import load_config, get_paths
from src.utils.logging import get_logger
from src.utils.mlflow_helpers import configure_mlflow
from src.data.loader import load_engineered_train

# Config
cfg = load_config()
paths = get_paths(cfg)
SEED = cfg['project']['random_seed']
TARGET = cfg['project']['target_col']
ID_COL = cfg['project']['id_col']
np.random.seed(SEED)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 100

# Initialise MLflow against the same backend Step 2 used
configure_mlflow(experiment='loan-default-baseline')

# Find the most recent baseline run with PR-AUC logged. This is the
# "current production model" the audit will run against.
runs = mlflow.search_runs(
    experiment_names=['loan-default-baseline'],
    filter_string='attributes.status = "FINISHED" and metrics.pr_auc > 0',
    order_by=['attributes.start_time DESC'],
    max_results=1,
)
assert not runs.empty, 'No FINISHED Step 2 runs found — re-run Step 2 first.'
run_id = runs.iloc[0]['run_id']
print(f'Auditing run: {run_id}')
print(f'  Logged PR-AUC: {runs.iloc[0]["metrics.pr_auc"]:.4f}')

# Load the model bundle
model = mlflow.xgboost.load_model(f'runs:/{run_id}/model')
preprocessor = joblib.load(paths.artifacts / 'preprocessor.joblib')
with open(paths.artifacts / 'feature_names.json') as fh:
    feature_names = json.load(fh)

# Reproduce Step 2's stratified split exactly
df_fe = load_engineered_train()
X = df_fe.drop(columns=[TARGET, ID_COL])
y = df_fe[TARGET].astype('int32')
ids = df_fe[ID_COL]

X_train, X_val, y_train, y_val, ids_train, ids_val = train_test_split(
    X, y, ids,
    test_size=0.20,
    stratify=y,
    random_state=SEED,
)
X_val_mat = preprocessor.transform(X_val)
print(f'\nValidation set: {X_val_mat.shape[0]:,} rows × {X_val_mat.shape[1]} features')

log = get_logger('step3')
log.info('Step 3 environment ready. Auditing run %s', run_id)

In [ ]:
from src.audit.shap_explain import compute_shap_values

# This takes 30-90 seconds depending on your CPU. TreeSHAP is exact —
# every value is exact attribution, not Monte Carlo approximation.
shap_values, expected_value = compute_shap_values(
    booster=model,
    X=X_val_mat,
    feature_names=feature_names,
)

print(f'SHAP values shape: {shap_values.shape}')
print(f'Expected value (baseline logit): {expected_value:.4f}')

# Additivity check — for a random row, sum of SHAPs + expected_value should
# equal the model's raw output (margin) for that row. If it doesn't, the
# SHAP computation is broken.
sample_idx = 42
dmat = xgb.DMatrix(pd.DataFrame(X_val_mat[sample_idx:sample_idx+1], columns=feature_names))
raw_margin = float(model.predict(dmat, output_margin=True)[0])
shap_sum = float(shap_values[sample_idx].sum() + expected_value)
print(f'\nAdditivity check on row {sample_idx}:')
print(f'  Model output (margin):   {raw_margin:.6f}')
print(f'  SHAP sum + expected:     {shap_sum:.6f}')
print(f'  Difference:              {abs(raw_margin - shap_sum):.2e}')
assert abs(raw_margin - shap_sum) < 1e-4, 'SHAP additivity broken'
print('Additivity verified.')

In [ ]:
from src.audit.shap_explain import top_features_by_mean_abs_shap, plot_top_features_bar

# Rank features by global importance.
top_features = top_features_by_mean_abs_shap(
    shap_values=shap_values,
    feature_names=feature_names,
    top_n=20,
)
print('Top 20 features by mean |SHAP|:')
display(top_features)

# Bar chart visualisation.
fig = plot_top_features_bar(top_features, title='Top 20 features by mean |SHAP|')
plt.show()

In [ ]:
from src.audit.shap_explain import plot_shap_summary

# Beeswarm summary — every point is one applicant on one feature.
fig = plot_shap_summary(
    shap_values=shap_values,
    X=X_val_mat,
    feature_names=feature_names,
    max_display=15,
    title='SHAP summary (validation set)',
)
plt.show()

In [ ]:
from src.models.train import predict_proba
from src.audit.shap_explain import plot_shap_waterfall, per_applicant_reason_codes

# Get model probabilities on the validation set.
y_val_pred = model.predict(xgb.DMatrix(pd.DataFrame(X_val_mat, columns=feature_names)))

# Find a correctly predicted high-risk applicant — true defaulter with a
# high model probability.
val_df = X_val.reset_index(drop=True).copy()
val_df['_target'] = y_val.reset_index(drop=True).values
val_df['_score'] = y_val_pred

high_risk = (val_df[(val_df['_target'] == 1) & (val_df['_score'] > 0.5)]
             .sort_values('_score', ascending=False))
print(f'Candidates (true defaulters with score > 0.5): {len(high_risk):,}')

# Pick one. Position 0 = highest-scoring; pick something a bit lower so the
# explanation is interesting (the very highest are often degenerate cases).
denied_idx = high_risk.index[5]
print(f'\nSelected applicant at val row index: {denied_idx}')
print(f'  Model score: {val_df.loc[denied_idx, "_score"]:.4f}')
print(f'  Actual outcome: {"defaulted" if val_df.loc[denied_idx, "_target"] == 1 else "repaid"}')

# Show key applicant inputs in human-readable form.
applicant_cols = ['AGE_YEARS', 'EMPLOYED_YEARS', 'AMT_INCOME_TOTAL', 'AMT_CREDIT',
                  'CREDIT_INCOME_RATIO', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
                  'CODE_GENDER', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS']
applicant_cols = [c for c in applicant_cols if c in val_df.columns]
print('\nApplicant snapshot:')
print(val_df.loc[denied_idx, applicant_cols].to_string())

# Waterfall: per-feature contribution to this specific decision.
fig = plot_shap_waterfall(
    shap_values=shap_values,
    expected_value=expected_value,
    X=X_val_mat,
    feature_names=feature_names,
    idx=denied_idx,
    max_display=12,
    title=f'High-risk applicant — SHAP waterfall (score = {val_df.loc[denied_idx, "_score"]:.3f})',
)
plt.show()

# Reason codes as a flat table — the adverse-action-notice format.
print('\nTop 5 reason codes for this decision:')
reasons = per_applicant_reason_codes(shap_values, feature_names, denied_idx, top_n=5)
display(reasons)

In [ ]:
# Mirror of Cell 5: find a true non-defaulter with a low model score.
low_risk = (val_df[(val_df['_target'] == 0) & (val_df['_score'] < 0.05)]
            .sort_values('_score', ascending=True))
print(f'Candidates (true non-defaulters with score < 0.05): {len(low_risk):,}')

approved_idx = low_risk.index[5]   # again, skip the very extreme tail
print(f'\nSelected applicant at val row index: {approved_idx}')
print(f'  Model score: {val_df.loc[approved_idx, "_score"]:.4f}')
print(f'  Actual outcome: {"defaulted" if val_df.loc[approved_idx, "_target"] == 1 else "repaid"}')

# Snapshot of the same human-readable columns.
print('\nApplicant snapshot:')
print(val_df.loc[approved_idx, applicant_cols].to_string())

# Waterfall.
fig = plot_shap_waterfall(
    shap_values=shap_values,
    expected_value=expected_value,
    X=X_val_mat,
    feature_names=feature_names,
    idx=approved_idx,
    max_display=12,
    title=f'Low-risk applicant — SHAP waterfall (score = {val_df.loc[approved_idx, "_score"]:.3f})',
)
plt.show()

# Reason codes for the approval decision.
print('\nTop 5 reason codes for this decision:')
reasons_approved = per_applicant_reason_codes(shap_values, feature_names, approved_idx, top_n=5)
display(reasons_approved)

In [ ]:
from src.audit.fairness import metrics_by_segment, disparity_summary, plot_segment_metric

# Pull the protected attribute from the original (pre-preprocessing) X_val.
# Important: this is for AUDIT only. The model was trained on the encoded
# CODE_GENDER_F / CODE_GENDER_M columns — see remediation note below.
gender = X_val.reset_index(drop=True)['CODE_GENDER']
print('Gender distribution in validation set:')
print(gender.value_counts(dropna=False))

# Sliced metrics — every column is a per-group fairness measurement.
gender_metrics = metrics_by_segment(
    y_true=y_val.values,
    y_score=y_val_pred,
    segments=gender,
    decision_threshold=0.5,
    precision_target=0.50,
)
print('\nMetrics by gender:')
display(gender_metrics)

# Four-fifths check on approval rate (the headline fair-lending metric).
gender_disparity = disparity_summary(
    gender_metrics,
    reference_segment=None,    # auto-picks the largest segment as reference
    metric_col='approval_rate',
)
print(f'\nFour-fifths-rule check (reference: largest segment = '
      f'{gender_disparity.attrs["reference_segment"]}):')
display(gender_disparity[['segment', 'n', 'approval_rate',
                          'disparity_ratio', 'four_fifths_violation']])

# Plots: approval rate side-by-side, recall side-by-side.
for metric, title in [
    ('approval_rate', 'Approval rate by gender (at threshold 0.5)'),
    ('recall_at_p50', 'Recall @ 50% precision by gender'),
    ('mean_score', 'Mean predicted score by gender (calibration proxy)'),
]:
    fig = plot_segment_metric(gender_metrics, metric, title=title)
    plt.show()

In [ ]:
# Pick a realistic operating threshold: reject the top 15% riskiest scores
# (an approval rate of ~85% — typical for auto-decisioning fintech).
operating_threshold = float(np.quantile(y_val_pred, 0.85))
print(f'Operating threshold (85th pctile of score): {operating_threshold:.4f}')
print(f'Overall rejection rate at this threshold: '
      f'{(y_val_pred >= operating_threshold).mean():.2%}')

# Re-run the fairness audit at the realistic threshold.
gender_metrics_real = metrics_by_segment(
    y_true=y_val.values,
    y_score=y_val_pred,
    segments=gender,
    decision_threshold=operating_threshold,
    precision_target=0.50,
)
print('\nMetrics by gender at the operating threshold:')
display(gender_metrics_real)

gender_disparity_real = disparity_summary(
    gender_metrics_real,
    reference_segment=None,
    metric_col='approval_rate',
)
print(f'\nFour-fifths-rule check at operating threshold:')
display(gender_disparity_real[['segment', 'n', 'approval_rate',
                                'disparity_ratio', 'four_fifths_violation']])

fig = plot_segment_metric(
    gender_metrics_real, 'approval_rate',
    title=f'Approval rate by gender (operating threshold = {operating_threshold:.3f})',
)
plt.show()

NOTE: My fairness audit caught a 0.884 disparity ratio on gender, passing the four-fifths rule by only 4 percentage points, with **CODE_GENDER** directly in the model. The remediation would be to retrain without protected attributes and re-audit. The framework's value is that it surfaced the issue before deployment, which is exactly what compliance teams need.

In [ ]:
from src.audit.fairness import income_decile

# Bucket validation applicants into 10 income deciles (D1 = lowest income).
income_d = income_decile(X_val.reset_index(drop=True)['AMT_INCOME_TOTAL'])
print('Decile counts:')
print(income_d.value_counts().sort_index())

# Run the audit at the same operating threshold from Cell 7b.
income_metrics = metrics_by_segment(
    y_true=y_val.values,
    y_score=y_val_pred,
    segments=income_d,
    decision_threshold=operating_threshold,
    precision_target=0.50,
)
print('\nMetrics by income decile:')
display(income_metrics)

# Four-fifths check, with the top decile (D10) as reference — the "best
# case" cohort whose treatment we're comparing the others against.
income_disparity = disparity_summary(
    income_metrics,
    reference_segment='D10',
    metric_col='approval_rate',
)
print('\nApproval-rate disparity vs top income decile (D10):')
display(income_disparity[['segment', 'n', 'approval_rate',
                          'disparity_ratio', 'four_fifths_violation']])

# Bar chart of approval rate across deciles.
fig = plot_segment_metric(
    income_metrics, 'approval_rate',
    title=f'Approval rate by income decile (threshold = {operating_threshold:.3f})',
)
plt.show()

fig = plot_segment_metric(
    income_metrics, 'base_rate',
    title='Actual default rate by income decile',
)
plt.show()

In [ ]:
# Save audit artifacts to the SAME run the model came from. This versions
# the audit with the model — when compliance asks "was THIS model audited?"
# the answer is one MLflow query away.
with mlflow.start_run(run_id=run_id):
    # ── Global SHAP importance ──────────────────────────────────────────
    fig = plot_top_features_bar(top_features, title='Top 20 features by mean |SHAP|')
    mlflow.log_figure(fig, 'audit/shap_global_importance.png')

    fig = plot_shap_summary(shap_values, X_val_mat, feature_names,
                            max_display=15, title='SHAP summary (validation)')
    mlflow.log_figure(fig, 'audit/shap_summary_beeswarm.png')

    # ── Gender fairness ─────────────────────────────────────────────────
    fig = plot_segment_metric(gender_metrics_real, 'approval_rate',
                              title='Approval rate by gender (operating threshold)')
    mlflow.log_figure(fig, 'audit/fairness_gender_approval.png')

    # Tables as CSVs — small, queryable, language-agnostic.
    gender_disparity_real.to_csv(paths.artifacts / 'gender_fairness.csv', index=False)
    mlflow.log_artifact(str(paths.artifacts / 'gender_fairness.csv'),
                        artifact_path='audit')

    # ── Income-decile fairness ──────────────────────────────────────────
    fig = plot_segment_metric(income_metrics, 'approval_rate',
                              title='Approval rate by income decile')
    mlflow.log_figure(fig, 'audit/fairness_income_approval.png')

    income_disparity.to_csv(paths.artifacts / 'income_fairness.csv', index=False)
    mlflow.log_artifact(str(paths.artifacts / 'income_fairness.csv'),
                        artifact_path='audit')

    # ── Headline audit metrics, queryable from the runs table ───────────
    mlflow.log_metrics({
        'audit_gender_disparity_M_vs_F': float(
            gender_disparity_real.loc[gender_disparity_real['segment'] == 'M',
                                       'disparity_ratio'].iloc[0]),
        'audit_income_disparity_D1_vs_D10': float(
            income_disparity.loc[income_disparity['segment'] == 'D1',
                                  'disparity_ratio'].iloc[0]),
        'audit_operating_threshold': operating_threshold,
    })

    # ── Tags so the run is filterable as "audited" ──────────────────────
    mlflow.set_tags({
        'audited': 'true',
        'audit_step': '3-shap-fairness',
        'shap_method': 'TreeSHAP',
        'fairness_segments': 'gender,income_decile',
    })

print(f'Audit logged to run: {run_id}')
print(f'View in UI: experiments → loan-default-baseline → click run → Artifacts tab → audit/')

## Step 3 — Summary

Built a TreeSHAP explainability layer and a fairness audit over the Step 2 XGBoost model. Computed exact SHAP values on the 61,503-row validation set in under a minute, verified additivity to floating-point precision, and produced both global feature importance and per applicant reason codes. Audited the model along two segments  gender (legally protected) and income decile (economic-axis proxy), using approval rate disparity at a realistic 15% rejection threshold (not the trivial 0.5 default). Attached every artifact to the same MLflow run as the model, so the audit is versioned with the thing it audits.

**Key findings:**

- **Gender:** disparity ratio M/F = 0.884. Technically passes the four-fifths rule, but in the yellow zone, any fair lending team would investigate. `CODE_GENDER_F` and `CODE_GENDER_M` are directly in the model;  a real fintech would retrain without these and reaudit.
- **Income decile:** all disparity ratios above 0.92 (D1 vs D10 = 0.952). The model is calibrated to the base rate gradient, not to income directly. Clean on this axis.
- **Global SHAP importance:** dominated by `EXT_SOURCE_*` features and the engineered ratios (`CREDIT_INCOME_RATIO`, `AGE_YEARS`). Same features a credit officer would reach for instinctively,  the model is finding signals where it should.
- **Per-applicant reason codes:** symmetric between approved and denied cases,  the features pushing one direction in a denial push the other direction in an approval. Stability is the precondition for legally defensible adverse-action notices.

**Three calls worth flagging:**

1. **TreeSHAP, not LIME or KernelSHAP.** Exact attribution is non-negotiable for compliance  auditors, who don't accept "approximately attributed" reason codes. TreeSHAP is also fast enough to run the full validation set; subsampling would hide tail behavior.
2. **Re-audited at a realistic operating threshold, not the default 0.5.** The original 0.5 threshold trivially passed every fairness check because almost no one is rejected. Moving the threshold to the 85th percentile (15% rejection rate) revealed the real gender disparity.
3. **Audit artifacts logged to the same MLflow run as the model.** A model from May 21 is auditable on May 21; in three months, when compliance asks "was THIS model audited?", the answer is one MLflow query away. Versioning the audit separately would let the two drift.

**Handoff to Step 4 (Evidently drift detection):** the model URI, the validation matrix, and a documented baseline operating threshold. Step 4 will simulate six months of production scoring and watch for both data drift (input distribution shift) and performance drift (PR-AUC degrading silently).